# FSL-SAGE on Colab GPU

Clones this repo from GitHub, installs dependencies, and runs a quick GPU smoke test
(see `docs/part0-mnist-smoke-test.md` for the equivalent CPU run this mirrors).

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better).

Datasets (`datas/`) and run outputs (`saves/`) are stored on your Google Drive so they
persist across Colab session resets instead of re-downloading/re-running every time.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change BRANCH once this work lands on master.
REPO_URL = "https://github.com/juniorfelix998/FSL-SAGE.git"
BRANCH = "ft/add-mnist"

WORKSPACE = "/content/drive/MyDrive/fsl-sage-colab"
REPO_DIR = f"{WORKSPACE}/FSL-SAGE"

import os
os.makedirs(WORKSPACE, exist_ok=True)

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

In [ ]:
# ---------------------------------------------------------------------------
# Run parameters. Everything below reads these, so moving from a plumbing check
# to a real benchmark run is an edit HERE, not in fifteen separate cells.
# ---------------------------------------------------------------------------

# Which cut to use -- "shallow", "middle" (default, this harness's original
# behavior), or "deep". See CLAUDE.md's "Cuts: early/middle/late" benchmark
# dimension and src/hydra_config/cut/*.yaml.
CUT = "middle"

# Override the number of clients (harness default is 10) -- set to an int
# (e.g. 2) or leave as None to use the default.
NUM_CLIENTS = None

# ROUNDS=3 is a PIPELINE CHECK, not a reportable result. The README/config.yaml
# default for a real run is 200, over 3 seeds (CLAUDE.md's protocol).
ROUNDS = 3
SEED = 200
SEEDS = [200]          # e.g. [200, 201, 202] for the 3-seed protocol

# ho_sfl and mu_splitfed process exactly ONE batch per client per round --
# faithful to their reference, which draws next(loader) each round. Every other
# method here takes one optimizer step per BATCH, so on MNIST with 10 clients
# (6000 samples each, batch 256 -> 24 batches) an equal ROUNDS budget hands
# these two 24x fewer updates, which is why they otherwise score near chance.
# The comm columns stay comparable regardless: Comm-to-target is read off at a
# fixed ACCURACY, not at a fixed round.
ZO_ROUND_MULTIPLIER = 24
ZO_ROUNDS = ROUNDS * ZO_ROUND_MULTIPLIER

# Accuracy at which Comm-to-target (the ranked number) is read off.
TARGET_ACC = 0.97

# Communication budget that early-stops a run. MU-SplitFed at 10 clients burns
# ~890 MB/round, so the 200 GiB default truncates it at ~230 rounds -- raise
# this whenever you raise ROUNDS.
COMM_THRESHOLD_MB = 204800

# Per-side (client vs. server) peak-memory instrumentation. Costs a few percent;
# set False only for a pure throughput measurement.
MEASURE_MEMORY = True

# ---------------------------------------------------------------------------
NUM_CLIENTS_OVERRIDE = f"num_clients={NUM_CLIENTS}" if NUM_CLIENTS is not None else ""
NUM_CLIENTS_LIST_ARG = f"--num_clients_list {NUM_CLIENTS}" if NUM_CLIENTS is not None else ""
COMMON = (
    f"model=resnet18 dataset=mnist cut={CUT} seed={SEED} "
    f"target_acc={TARGET_ACC} comm_threshold_mb={COMM_THRESHOLD_MB} "
    f"measure_memory={str(MEASURE_MEMORY).lower()} {NUM_CLIENTS_OVERRIDE}"
)
SEEDS_ARG = f"--seeds {' '.join(str(s) for s in SEEDS)}"
MEASURE_MEMORY_ARG = f"--measure_memory {str(MEASURE_MEMORY).lower()}"
print(COMMON)
print(SEEDS_ARG, MEASURE_MEMORY_ARG)

In [ ]:
# src/main.py resolves datasets/saves as '../datas' and '../saves' relative to
# src/, so point those at persistent Drive folders instead of Colab's ephemeral disk.
DRIVE_DATAS = f"{WORKSPACE}/datas"
DRIVE_SAVES = f"{WORKSPACE}/saves"
os.makedirs(DRIVE_DATAS, exist_ok=True)
os.makedirs(DRIVE_SAVES, exist_ok=True)

for name, target in (("datas", DRIVE_DATAS), ("saves", DRIVE_SAVES)):
    link = f"{REPO_DIR}/{name}"
    if os.path.islink(link) or os.path.exists(link):
        continue
    os.symlink(target, link)

In [ ]:
# torch/torchvision are pinned to Colab's OWN already-installed versions (not the
# repo's local conda_env.yaml pin of torch==2.5.1) so pip has no reason to touch them --
# swapping torch pulls in a different CUDA toolkit than the one Colab's preinstalled
# RAPIDS stack (cuml/cudf/libraft/libcuvs/cuda-python) was built against, which is what
# caused the wall of "cuda-toolkit ... incompatible" resolver errors. requests is bumped
# to 2.32.4 to match what google-colab/google-adk already require, for the same reason.
# Check !python -c "import torch, torchvision; print(torch.__version__, torchvision.__version__)"
# on a fresh runtime if these ever drift from what Colab ships.
!pip install -q torch==2.13.0 torchvision==0.28.0 hydra-core==1.3.2 hydra-joblib-launcher==1.2.0 \
  omegaconf==2.3.0 wandb==0.19.3 numpy==2.1.3 pandas==2.2.3 scipy==1.14.1 matplotlib==3.9.2 \
  h5py==3.12.1 pyyaml==6.0.2 tqdm==4.67.0 requests==2.32.4 pillow==11.0.0 prettytable==3.12.0 \
  joblib==1.4.2 antlr4-python3-runtime==4.9.3 gitpython==3.1.43

# NOTE: torch==2.13.0 is also the version the per-side memory instrumentation
# (torch.autograd.graph.saved_tensors_hooks, see src/utils/memory.py) was
# validated against.

## Correctness gate: check the measurement layer before spending GPU hours

`test/check_accounting.py` asserts the communication counters against
**closed-form** expectations per method (e.g. SplitFedv2 must charge
`2 x sum(batch x C x H x W x 4)` across the cut; CSE-FSL must charge zero
returned-gradient bytes) and checks the memory meter's invariants (exact byte
counts on known shapes, exactly zero retained activations under
`torch.no_grad()`).

It runs in seconds. A sweep runs for hours, and a silently-wrong byte counter
invalidates every number the sweep produces -- so run this **first**, and note
that it is worth re-running here on the GPU, because cuDNN saves a different
set of intermediates than the CPU kernels do.

In [ ]:
%cd {REPO_DIR}
!python test/check_accounting.py

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds={ROUNDS} save=False device=cuda {COMMON}

## Running a single method standalone

Supported `algorithm=` keys: `fed_avg`, `sl_multi_server` (SplitFedv1),
`sl_single_server` (SplitFedv2), `vanilla_sl`, `cse_fsl`, `fsl_sage`, `ho_sfl`,
`mu_splitfed`, `dsl_aux`, `han_locloss`, `fedsplitx`, `hosl`, `locfedmix_sl`.

**`mu_splitfed` provenance note:** this is HKU-WILL-Lab/HO-SFL's own third-party
CV/ResNet18 reimplementation of MU-SplitFed, not the original Johnny-Zip/MU-SplitFed
authors' code (their published repo is LLM-only and non-functional as published --
see `src/algos/mu_splitfed.py` for details). Treat any MU-SplitFed numbers accordingly.

Each cell below runs one method for `rounds=3` (a quick sanity check, not a real
result) so you can test any single method on its own without running the full sweep.

**`dsl_aux` provenance note:** AI-assisted no-code reimplementation of DSL-Aux (arXiv:2601.19261), adapted from a partial third-party reference (juniorfelix998/sl-fl-dgl) -- not validated against the paper's own reported numbers. It is also non-federated (single client/server split, no weight aggregation); `num_clients=1` is its paper-faithful setting, though the cell below runs it with the notebook's usual default like every other method for consistency.
**`vanilla_sl` note:** the original, non-federated sequential split-learning
protocol (Gupta & Raskar 2018) -- one shared client model and one shared server
model, no FedAvg step (there's only one copy of each to begin with).

**`han_locloss` provenance note:** AI-assisted no-code reimplementation of Han et
al., "Accelerating FL with SL on Locally Generated Losses" (FL-ICML 2021) -- not
validated against the paper's own reported numbers.

**`fedsplitx` provenance note:** AI-assisted no-code reimplementation of FedSplitX
(arXiv:2310.14579), run at a single shared cut (this benchmark's protocol) rather
than the paper's own multiple simultaneous depth-levels -- not validated against
the paper's own reported numbers.

**`hosl` provenance note:** AI-assisted no-code reimplementation of HOSL
(arXiv:2601.10940) -- distinct from `ho_sfl` above (different paper, confirmed
different arXiv id); not validated against the paper's own reported numbers.

**`locfedmix_sl` provenance note:** AI-assisted no-code reimplementation of
LocFedMix-SL (ACM WWW 2022) -- not validated against the paper's own reported
numbers.

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds={ROUNDS} save=False device=cuda algorithm=fed_avg {COMMON}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds={ROUNDS} save=False device=cuda algorithm=sl_multi_server {COMMON}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds={ROUNDS} save=False device=cuda algorithm=sl_single_server {COMMON}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds={ROUNDS} save=False device=cuda algorithm=cse_fsl {COMMON}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds={ROUNDS} save=False device=cuda algorithm=fsl_sage {COMMON}

In [ ]:
# ho_sfl does one batch per client per round -- see ZO_ROUNDS above.
%cd {REPO_DIR}/src
!python main.py rounds={ZO_ROUNDS} save=False device=cuda algorithm=ho_sfl {COMMON}

In [ ]:
# mu_splitfed does one batch per client per round -- see ZO_ROUNDS above.
%cd {REPO_DIR}/src
!python main.py rounds={ZO_ROUNDS} save=False device=cuda algorithm=mu_splitfed {COMMON}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds={ROUNDS} save=False device=cuda algorithm=dsl_aux {COMMON}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds={ROUNDS} save=False device=cuda algorithm=vanilla_sl {COMMON}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds={ROUNDS} save=False device=cuda algorithm=han_locloss {COMMON}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds={ROUNDS} save=False device=cuda algorithm=fedsplitx {COMMON}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds={ROUNDS} save=False device=cuda algorithm=hosl {COMMON}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds={ROUNDS} save=False device=cuda algorithm=locfedmix_sl {COMMON}

## Running everything: sweep + measurement table + plots in one command

`run_mnist_benchmark.py` is the "one main" entry point: it sweeps all 12 tabled
methods (SplitFedv1, SplitFedv2, Vanilla-SL, CSE-FSL, FSL-SAGE, HO-SFL,
MU-SplitFed, DSL-Aux, Han-et-al, FedSplitX, HOSL, LocFedMix-SL) across both
MNIST distributions (IID and Dirichlet alpha=0.5), builds the measurement
table, and generates accuracy/communication-load plots -- all from one command.

### What the table columns mean

| Column | Meaning |
|---|---|
| Comm-cut / Comm-weights / Comm-total (MB) | cumulative bytes at the final round, split into activation+gradient traffic across the cut vs. model-weight/aggregation traffic |
| **Comm-to-target (MB)** | **the ranked number**: bytes needed to first reach `TARGET_ACC`. Cumulative bytes at the final round are round-count dependent -- they penalise a method that ran longer and reward one that converges slowly. `n/a` means the method never reached the target, which is shown as such rather than as a number |
| Acc (%) | final-round test accuracy |
| Client mem (MB) | peak memory ONE client device must provide: params + grads + optimizer state + peak retained activations |
| **Held-across-cut (MB)** | client autograd bytes still live while the server runs -- the memory cost of sending a gradient back. `> 0` for synchronous split learning (SplitFedv1/v2, Vanilla-SL, DSL-Aux, LocFedMix-SL), exactly `0` for the decoupled methods |
| Server mem (MB) | the same decomposition for the server host (which really does hold N replicas for the multi-server methods) |

Every value is a mean +/- std across `SEEDS` when more than one seed is run.
The table also **refuses to pretend cells are comparable** when they are not:
it prints a `NOT COMPARABLE` warning if cells differ in round count, client
count, cut, or device.

### One result that is easy to misread

"BP-free uses less client memory" holds only for the **zeroth-order** family
(HOSL, HO-SFL, MU-SplitFed), which run the client forward under
`torch.no_grad()` and retain ~0 activation bytes. The auxiliary-model family
(CSE-FSL, FSL-SAGE, DSL-Aux) builds the client graph *and* runs an auxiliary
head while it is alive, so its **Client mem is expected to be HIGHER** than
SplitFed's. What those methods actually save is communication and the stall --
which is why Held-across-cut is reported as a separate column. Two axes, not
one.

### Scale

Runs at the `CUT` cell's cut by default; pass `--cuts shallow middle deep`
(3x the sweep cost) to compare cut-depth sensitivity -- each cut gets its own
table (`benchmark_table_mnist_<cut>.txt`) and plots directory
(`plots_mnist_<cut>/`). Similarly `--num_clients_list 2 10 100` sweeps client
counts (each gets a `_nc<N>` suffix).

**At `ROUNDS = 3` this is a pipeline check, not a reportable result.** For real
numbers raise `ROUNDS` (README/`config.yaml` default: 200) and set `SEEDS` to
three seeds. At that scale a single free-tier Colab session likely will not
finish in one sitting -- use `--methods` to re-invoke for a subset across
several sessions; each sweep run is additive (new timestamped folders under
`saves/`), nothing gets overwritten.

In [ ]:
%cd {REPO_DIR}/inference
!python run_mnist_benchmark.py --rounds {ROUNDS} --device cuda --cuts {CUT} \
  {SEEDS_ARG} --target_acc {TARGET_ACC} \
  --zo_round_multiplier {ZO_ROUND_MULTIPLIER} \
  --comm_threshold_mb {COMM_THRESHOLD_MB} \
  {MEASURE_MEMORY_ARG} {NUM_CLIENTS_LIST_ARG}

In [ ]:
# Inline display of the table + plots, so results are visible without downloading
# anything from Drive. Matches benchmark_table.py's cut/num_clients-aware naming:
# the 'middle' cut + default num_clients keeps the original filenames; shallow/deep
# add a `_<cut>` suffix and an explicit NUM_CLIENTS adds a `_nc<N>` suffix.
import glob
from IPython.display import Image, display

table_suffix = '' if CUT == 'middle' else f'_{CUT}'
plots_dir_name = 'plots_mnist' if CUT == 'middle' else f'plots_mnist_{CUT}'
if NUM_CLIENTS is not None:
    table_suffix += f'_nc{NUM_CLIENTS}'
    plots_dir_name += f'_nc{NUM_CLIENTS}'

print(open(f"{REPO_DIR}/inference/benchmark_table_mnist{table_suffix}.txt").read())

for png in sorted(glob.glob(f"{REPO_DIR}/inference/{plots_dir_name}/**/*.png", recursive=True)):
    print(png)
    display(Image(filename=png))

## Running a custom experiment

Use the README's Hydra override syntax to run any method/model/dataset
combination, e.g.:

```python
!python main.py algorithm=fsl_sage model=resnet18 dataset=cifar10 \\
  dataset.distribution=noniid_dirichlet dataset.alpha=0.5 cut=shallow \\
  num_clients=20 save=False device=cuda
```

Add `cut=shallow` / `cut=middle` / `cut=deep` to pick the client/server split
point (default `middle`); see `src/hydra_config/cut/*.yaml`. Add
`num_clients=<N>` to override the client count (default 10). Measurement knobs:
`target_acc=`, `measure_memory=`, `mem_probe_batches=`, `comm_threshold_mb=`.

Results land under
`saves/<algorithm>/<model>/<cut>/<dataset>-<distribution>/.../results.json`
(the client count is folded into that final path segment), which via the
symlink above is actually on your Drive at `fsl-sage-colab/saves/...` so it
survives runtime resets.

Each `results.json` carries:

- `comm_load`, `comm_load_cut`, `comm_load_weights` -- per-round cumulative bytes
- `comm_breakdown` -- the same bytes split by category (`cut.act_up`,
  `cut.grad_down`, `cut.labels_up`, `cut.scalar_down`, `weights.client_up`,
  `weights.aux_down`, `weights.scalars`, ...). This is what makes a ranking
  auditable: a method claiming to be BP-free must show `cut.grad_down == 0`
- `comm_to_target` / `comm_cut_to_target` / `rounds_to_target` -- `None` if the
  target accuracy was never reached
- `peak_client_mem_mb`, `peak_server_mem_mb`,
  `client_mem_held_across_cut_mb`, plus the `param` / `grad` / `optim` / `act`
  breakdown per side and a per-client list
- `peak_memory_mb` (= `peak_process_mem_mb`) -- whole-process RSS. A
  **diagnostic only**: it is dominated by the interpreter, torch, the dataset in
  RAM and all simulated clients at once, and is not a per-device metric
- `run_manifest` -- git commit + dirty flag, config SHA, seed, rounds,
  num_clients, cut, dataset, dtype, device